# One Look — Stage 2: inspect images, create labels and split

Run this after `01_setup_data.ipynb` has downloaded the dataset. All output is saved in Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install rasterio scikit-learn joblib
from pathlib import Path
import os, json, csv
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt
DATA = Path('/content/drive/MyDrive/one-look/data/HandLabeled')
OUT = Path('/content/drive/MyDrive/one-look/data')
assert (DATA/'LabelHand').exists(), 'Run 01_setup_data.ipynb first.'

## 1. Look at one satellite image and its masks
The true-colour image is made from Sentinel-2 red, green and blue bands. The label mask marks water; the JRC mask marks permanent water such as rivers and lakes.

In [ ]:
cid = sorted(p.name.replace('_LabelHand.tif','') for p in (DATA/'LabelHand').glob('*_LabelHand.tif'))[0]
with rasterio.open(DATA/'S2Hand'/f'{cid}_S2Hand.tif') as ds: s2 = ds.read()
with rasterio.open(DATA/'LabelHand'/f'{cid}_LabelHand.tif') as ds: lab = ds.read(1)
jrc_path = DATA/'JRCWaterHand'/f'{cid}_JRCWaterHand.tif'
with rasterio.open(jrc_path) as ds: jrc = ds.read(1)
rgb = np.clip(s2[[3,2,1]].transpose(1,2,0)/3000,0,1)
fig, ax = plt.subplots(1,3,figsize=(14,4))
ax[0].imshow(rgb); ax[0].set_title(f'{cid} — satellite image')
ax[1].imshow(lab,vmin=-1,vmax=1); ax[1].set_title('Water label (-1 / 0 / 1)')
ax[2].imshow(jrc); ax[2].set_title('Permanent water (JRC)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 2. Create one label per image
A chip is ALERT (1) if non-permanent water covers more than 1% of its valid pixels. Otherwise it is NO ALERT (0). This is the project's initial rule, not a clinically or operationally validated threshold.

In [ ]:
THRESHOLD = 0.01
rows = []
for lp in sorted((DATA/'LabelHand').glob('*_LabelHand.tif')):
    cid = lp.name.replace('_LabelHand.tif','')
    with rasterio.open(lp) as ds: lab = ds.read(1)
    jp = DATA/'JRCWaterHand'/f'{cid}_JRCWaterHand.tif'
    with rasterio.open(jp) as ds: jrc = ds.read(1)
    valid = lab != -1
    if not valid.any(): continue
    flood = (lab == 1) & (jrc != 1)
    frac = float(flood[valid].sum()/valid.sum())
    rows.append({'chip_id':cid,'country':cid.split('_')[0],'flood_fraction':frac,'label':int(frac>THRESHOLD)})
df = pd.DataFrame(rows)
assert not df.empty, 'No valid labels created.'
print('Total chips:',len(df)); print('Class counts:'); display(df.label.value_counts().rename(index={0:'NO ALERT',1:'ALERT'}))
display(df.groupby('country').label.agg(chips='count', alerts='sum'))

## 3. Geographic train / validation / test split
All images from a country stay together, so nearby (near-duplicate) chips never leak from training into testing.
**Test = Bolivia + Ghana** (never seen in training; Bolivia alone has only 4 NO ALERT chips, too few to measure the
false-alarm rate). **Validation = Spain + Sri-Lanka** (only for choosing thresholds). **Train = the other 7 countries.**

In [ ]:
SPLIT = {'test': ['Bolivia', 'Ghana'], 'val': ['Spain', 'Sri-Lanka']}
SPLIT['train'] = sorted(set(df.country) - set(SPLIT['test']) - set(SPLIT['val']))
owner = {c: s for s, cs in SPLIT.items() for c in cs}
df['split'] = df.country.map(owner)
assert df.split.notna().all(), 'A country was not assigned.'
assert not (set(SPLIT['train']) & set(SPLIT['test'])), 'Country leakage detected.'
print(json.dumps(SPLIT, indent=2)); display(pd.crosstab(df.split, df.label, margins=True))
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT/'labels.csv', index=False)
(OUT/'splits.json').write_text(json.dumps(SPLIT, indent=2))
print('Saved labels.csv and splits.json to', OUT)